# Maze Crawler Episodes -- Advanced EDA

A multi-layer exploration of the **Maze Crawler** simulation ecosystem,
covering 24 days of daily episode data and individual JSON replay files.

**Analysis layers**

1. Macro -- daily manifest trends (submission counts, storage, scores)
2. Episode -- reward trajectories, action distributions, map topology
3. Aggregate -- multi-episode efficiency scores and score distributions
4. Strategic -- action Markov chains, early-game signals, crystal depletion


In [ ]:
import json
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# -- Dark research theme ------------------------------------------------
BG     = '#0F0F1A'
PANEL  = '#1A1A2E'
BORDER = '#2D2D4A'
CYAN   = '#00D4FF'
CORAL  = '#FF4757'
AMBER  = '#FFA502'
GREEN  = '#2ED573'
VIOLET = '#7B68EE'
PINK   = '#FF6EB4'
GOLD   = '#FFD166'
TEXT   = '#EAECEF'
MUTED  = '#A8B2C1'
PAL    = [CYAN, CORAL, AMBER, GREEN, VIOLET, PINK, GOLD]

plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    PANEL,
    'axes.edgecolor':    BORDER,
    'axes.labelcolor':   MUTED,
    'axes.titlecolor':   TEXT,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.titlepad':     14,
    'axes.labelsize':    11,
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'xtick.color':       MUTED,
    'ytick.color':       MUTED,
    'grid.color':        BORDER,
    'grid.alpha':        0.5,
    'grid.linewidth':    0.6,
    'text.color':        TEXT,
    'figure.dpi':        110,
    'figure.figsize':    (12, 6),
    'legend.facecolor':  PANEL,
    'legend.edgecolor':  BORDER,
    'legend.labelcolor': MUTED,
    'savefig.facecolor': BG,
})

pd.set_option('display.max_columns', None)
print('Dark theme ready. Palette:', PAL)


---
## 2. Daily trends from the index manifest

The `maze-crawler-episodes-index` dataset publishes a `manifest.csv` that
records per-day episode counts, raw byte totals, and aggregated scores.
We use this to track the competition's growth and the field's improvement over time.


In [ ]:
INDEX_DIR = '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-index'
manifest_path = os.path.join(INDEX_DIR, 'manifest.csv')

if os.path.exists(manifest_path):
    df_m = pd.read_csv(manifest_path)
    df_m['date'] = pd.to_datetime(df_m['date'])
    display(df_m.head())

    # -- Plot 1: episode count + storage volume (dual-axis, filled) --------
    fig, ax1 = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor(BG)

    ax1.fill_between(df_m['date'], df_m['episode_count'],
                     alpha=0.18, color=CYAN)
    ax1.plot(df_m['date'], df_m['episode_count'],
             color=CYAN, marker='o', linewidth=2.2, markersize=5,
             label='Episodes per day')
    ax1.set_ylabel('Episode count', color=CYAN)
    ax1.tick_params(axis='y', labelcolor=CYAN)
    ax1.set_xlabel('Date')

    ax2 = ax1.twinx()
    ax2.spines['right'].set_visible(True)
    ax2.spines['right'].set_color(BORDER)
    ax2.fill_between(df_m['date'], df_m['total_bytes'] / 1e9,
                     alpha=0.12, color=AMBER)
    ax2.plot(df_m['date'], df_m['total_bytes'] / 1e9,
             color=AMBER, marker='s', linewidth=2.2, markersize=5,
             label='Storage (GB)')
    ax2.set_ylabel('Storage (GB)', color=AMBER)
    ax2.tick_params(axis='y', labelcolor=AMBER)
    ax2.tick_params(colors=MUTED)

    ax1.set_title('Daily episode submissions and cumulative storage growth')
    handles = [
        mpatches.Patch(color=CYAN,  label='Episodes per day'),
        mpatches.Patch(color=AMBER, label='Storage (GB)'),
    ]
    ax1.legend(handles=handles, loc='upper left')
    plt.tight_layout()
    plt.show()

    # -- Plot 2: score evolution with band ---------------------------------
    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor(BG)

    ax.fill_between(df_m['date'],
                    df_m['top_avg_score'], df_m['median_avg_score'],
                    alpha=0.15, color=GREEN, label='Top -- median band')
    ax.plot(df_m['date'], df_m['top_avg_score'],
            color=GREEN, linewidth=2.4, marker='o', markersize=5,
            label='Top avg score')
    ax.plot(df_m['date'], df_m['median_avg_score'],
            color=VIOLET, linewidth=2.4, marker='s', markersize=5,
            linestyle='--', label='Median avg score')

    ax.set_xlabel('Date')
    ax.set_ylabel('Score')
    ax.set_title('Score evolution -- top vs median agents over 24 days')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('manifest.csv not found -- attach maze-crawler-episodes-index')


---
## 3. Reading simulation JSON replays

Each daily directory contains one JSON file per episode. We load a sample file
and unpack the top-level keys to understand the replay schema before diving deeper.


In [ ]:
BASE = '/kaggle/input/datasets/organizations/kaggle'
daily_paths = sorted(glob.glob(f'{BASE}/maze-crawler-episodes-2026-*'))

sample_files = glob.glob('*.json')
for dp in daily_paths:
    if os.path.exists(dp):
        sample_files.extend(glob.glob(os.path.join(dp, '*.json')))

episode_data = {}
if sample_files:
    with open(sample_files[0], 'r') as f:
        episode_data = json.load(f)
    print(f'Loaded: {os.path.basename(sample_files[0])}')
    print(f'Top-level keys: {list(episode_data.keys())}')

    statuses = episode_data.get('statuses', [])
    rewards  = episode_data.get('rewards', [])
    config   = episode_data.get('configuration', {})

    print('\n--- Match outcome ---')
    for i, (s, r) in enumerate(zip(statuses, rewards)):
        tag = 'WINNER' if r == max(rewards) else 'loser '
        print(f'  Agent {i}  [{tag}]  status={s}  reward={r}')

    print('\n--- Configuration ---')
    for k in ['episodeSteps', 'height', 'width',
              'crystalDensity', 'miningNodeDensity', 'energyPerTurn']:
        if k in config:
            print(f'  {k}: {config[k]}')
else:
    print('No JSON files found. Attach the daily episode datasets.')


---
## 4. Step-by-step telemetry

Each episode's `steps` array records the full game state at every turn.
We extract per-agent reward accumulation and display it as a trajectory,
annotating the winner at the final step.


In [ ]:
steps = episode_data.get('steps', [])
print(f'Total steps: {len(steps)}')

a0_rewards, a1_rewards = [], []
a0_actions, a1_actions = [], []

for step in steps:
    if len(step) > 0:
        r0 = step[0].get('reward') or 0
        a0_rewards.append(r0)
        a0_act = step[0].get('action') or {}
        a0_actions.extend(a0_act.values())
    if len(step) > 1:
        r1 = step[1].get('reward') or 0
        a1_rewards.append(r1)
        a1_act = step[1].get('action') or {}
        a1_actions.extend(a1_act.values())

if a0_rewards:
    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor(BG)

    xs = range(len(a0_rewards))
    ax.fill_between(xs, a0_rewards, alpha=0.12, color=CYAN)
    ax.fill_between(xs, a1_rewards, alpha=0.12, color=CORAL)
    ax.plot(xs, a0_rewards, color=CYAN,  linewidth=2.2, label='Agent 0')
    ax.plot(xs, a1_rewards, color=CORAL, linewidth=2.2, label='Agent 1')

    if a0_rewards and a1_rewards:
        final_step = len(a0_rewards) - 1
        ax.annotate(f'{a0_rewards[-1]:.1f}',
                    xy=(final_step, a0_rewards[-1]),
                    xytext=(-18, 8), textcoords='offset points',
                    color=CYAN, fontsize=10, fontweight='bold')
        ax.annotate(f'{a1_rewards[-1]:.1f}',
                    xy=(final_step, a1_rewards[-1]),
                    xytext=(-18, -18), textcoords='offset points',
                    color=CORAL, fontsize=10, fontweight='bold')

    ax.set_xlabel('Step')
    ax.set_ylabel('Accumulated reward')
    ax.set_title('Reward trajectory -- episode replay')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()
    plt.show()


### 4.1 Action distribution

How often does each agent issue each command type?
A horizontal grouped bar chart makes cross-agent comparison easy.


In [ ]:
if a0_actions or a1_actions:
    all_actions = sorted(set(a0_actions) | set(a1_actions))
    a0_counts = pd.Series(a0_actions).value_counts().reindex(all_actions, fill_value=0)
    a1_counts = pd.Series(a1_actions).value_counts().reindex(all_actions, fill_value=0)

    y = np.arange(len(all_actions))
    bar_h = 0.35

    fig, ax = plt.subplots(figsize=(12, max(4, len(all_actions) * 0.7)))
    fig.patch.set_facecolor(BG)

    bars0 = ax.barh(y + bar_h / 2, a0_counts.values, bar_h,
                    color=CYAN,  alpha=0.85, label='Agent 0')
    bars1 = ax.barh(y - bar_h / 2, a1_counts.values, bar_h,
                    color=CORAL, alpha=0.85, label='Agent 1')

    ax.set_yticks(y)
    ax.set_yticklabels(all_actions)
    ax.set_xlabel('Count')
    ax.set_title('Action frequency by agent -- single episode')
    ax.legend()
    ax.grid(True, axis='x')

    for bar in bars0:
        w = bar.get_width()
        if w > 0:
            ax.text(w + 1, bar.get_y() + bar.get_height() / 2,
                    str(int(w)), va='center', color=CYAN, fontsize=8)
    for bar in bars1:
        w = bar.get_width()
        if w > 0:
            ax.text(w + 1, bar.get_y() + bar.get_height() / 2,
                    str(int(w)), va='center', color=CORAL, fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print('No actions recorded in this episode.')


---
## 5. Multi-episode aggregate statistics

Parsing up to 200 episodes gives us enough data to study the distribution of
match outcomes, score margins, and game lengths.


In [ ]:
MAX_EP = 200
agg = []

valid = [f for f in sample_files if os.path.exists(f)][:MAX_EP]
for fp in valid:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        rew    = ep.get('rewards', [0, 0])
        nsteps = len(ep.get('steps', []))
        if len(rew) >= 2:
            agg.append({
                'steps':        nsteps,
                'a0':           rew[0],
                'a1':           rew[1],
                'winner':       max(rew),
                'loser':        min(rew),
                'margin':       abs(rew[0] - rew[1]),
                'a0_wins':      int(rew[0] > rew[1]),
            })
    except Exception:
        continue

df_agg = pd.DataFrame(agg)
print(f'Parsed {len(df_agg)} episodes')

if not df_agg.empty:
    display(df_agg.describe().round(2))

    fig = plt.figure(figsize=(14, 6))
    fig.patch.set_facecolor(BG)
    gs = GridSpec(1, 2, figure=fig, wspace=0.35)

    # Scatter: game length vs margin
    ax0 = fig.add_subplot(gs[0])
    sc = ax0.scatter(df_agg['steps'], df_agg['margin'],
                     c=df_agg['winner'], cmap='plasma',
                     s=45, alpha=0.65, edgecolors='none')
    cb = plt.colorbar(sc, ax=ax0)
    cb.set_label('Winner score', color=MUTED)
    cb.ax.yaxis.set_tick_params(color=MUTED)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color=MUTED)
    ax0.set_xlabel('Match steps')
    ax0.set_ylabel('Score margin')
    ax0.set_title('Game length vs margin of victory')
    ax0.grid(True)

    # KDE: winner vs loser score
    ax1 = fig.add_subplot(gs[1])
    sns.kdeplot(df_agg['winner'], fill=True, color=GREEN,
                alpha=0.35, linewidth=2, label='Winner score', ax=ax1)
    sns.kdeplot(df_agg['loser'],  fill=True, color=CORAL,
                alpha=0.35, linewidth=2, label='Loser score',  ax=ax1)
    ax1.set_xlabel('Final reward')
    ax1.set_ylabel('Density')
    ax1.set_title('Final score distribution -- winners vs losers')
    ax1.legend()
    ax1.grid(True, axis='y')

    plt.suptitle(f'Aggregate stats across {len(df_agg)} episodes',
                 color=TEXT, fontsize=13, y=1.02)
    plt.show()


### 5.1 Momentum tracking

Momentum -- the rolling mean of per-step reward delta -- reveals whether
agents snowball an early lead or mount comebacks. A positive momentum means
the agent is actively gaining score; a negative one means the opponent is.


In [ ]:
if len(a0_rewards) > 10:
    win = max(a0_rewards[-1], a1_rewards[-1]) if a1_rewards else a0_rewards[-1]
    w = max(5, int(len(a0_rewards) * 0.08))

    m0 = pd.Series(a0_rewards).diff().rolling(window=w, min_periods=1).mean()
    m1 = pd.Series(a1_rewards).diff().rolling(window=w, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor(BG)

    ax.fill_between(range(len(m0)), m0, 0, where=(m0 > 0),
                    alpha=0.15, color=CYAN,  interpolate=True)
    ax.fill_between(range(len(m0)), m0, 0, where=(m0 < 0),
                    alpha=0.15, color=CORAL, interpolate=True)
    ax.plot(m0, color=CYAN,  linewidth=2.2, label=f'Agent 0 momentum (MA={w})')
    ax.plot(m1, color=CORAL, linewidth=2.2, label=f'Agent 1 momentum (MA={w})')
    ax.axhline(0, color=BORDER, linewidth=1.2, linestyle='--')

    ax.set_xlabel('Step')
    ax.set_ylabel('Avg points gained per step')
    ax.set_title('Momentum swings -- identifying the exploitation phase')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print(f'Episode only has {len(a0_rewards)} steps -- too short for momentum.')


### 5.2 Environmental evolution

We track map-entity counts extracted from Agent 0's observation at each step:
global crystals, mining nodes, and active robots.


In [ ]:
crystals_t, nodes_t, robots_t = [], [], []

for step in steps:
    if step:
        obs = step[0].get('observation', {})
        crystals_t.append(len(obs.get('globalCrystals',    {})))
        nodes_t.append(   len(obs.get('globalMiningNodes', {})))
        robots_t.append(  len(obs.get('robots',            {})))

if crystals_t:
    fig, ax1 = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor(BG)

    ax1.fill_between(range(len(crystals_t)), crystals_t,
                     alpha=0.12, color=GREEN)
    ax1.plot(crystals_t, color=GREEN,  linewidth=2.2, label='Crystals')
    ax1.fill_between(range(len(nodes_t)), nodes_t,
                     alpha=0.12, color=AMBER)
    ax1.plot(nodes_t,   color=AMBER,   linewidth=2.2,
             linestyle='--', label='Mining nodes')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Item count')

    ax2 = ax1.twinx()
    ax2.spines['right'].set_visible(True)
    ax2.spines['right'].set_color(BORDER)
    ax2.plot(robots_t, color=VIOLET, linewidth=2.2,
             linestyle=':', label='Active robots')
    ax2.set_ylabel('Active robots', color=VIOLET)
    ax2.tick_params(axis='y', labelcolor=VIOLET, colors=VIOLET)

    lines1, labs1 = ax1.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper right')
    ax1.set_title('Map entity evolution over match duration')
    plt.tight_layout()
    plt.show()


### 5.3 Initial board topology

Visualising the opening layout -- crystal deposits, mining nodes, and robot
starting positions -- gives intuition about map geometry and initial resource proximity.


In [ ]:
def parse_coords(d):
    pts = []
    for k in d.keys():
        try:
            pts.append(tuple(map(int, k.split(','))))
        except ValueError:
            pass
    return pts

if steps:
    obs0 = steps[0][0].get('observation', {})
    crystals = parse_coords(obs0.get('globalCrystals',    {}))
    nodes    = parse_coords(obs0.get('globalMiningNodes', {}))

    robot_pts, robot_clr = [], []
    for r_id, r_data in obs0.get('robots', {}).items():
        if len(r_data) >= 3:
            robot_pts.append((r_data[1], r_data[2]))
            robot_clr.append(CYAN if str(r_data[0]) == '0' else CORAL)

    fig, ax = plt.subplots(figsize=(8, 8))
    fig.patch.set_facecolor(BG)

    if crystals:
        cx, cy = zip(*crystals)
        ax.scatter(cx, cy, color=GREEN,  marker='D', s=55,
                   alpha=0.75, label='Crystals', zorder=2)
    if nodes:
        nx, ny = zip(*nodes)
        ax.scatter(nx, ny, color=AMBER,  marker='s', s=90,
                   alpha=0.85, label='Mining nodes', zorder=3)
    if robot_pts:
        rx, ry = zip(*robot_pts)
        ax.scatter(rx, ry, c=robot_clr, marker='o', s=180,
                   edgecolors=TEXT, linewidths=1.5,
                   label='Robots', zorder=4)

    ax.invert_yaxis()
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate (0 = top)')
    ax.set_title('Opening board topology -- resources and robot spawn points')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.4)
    plt.tight_layout()
    plt.show()


---
## 6. Agent efficiency score

Raw final reward does not tell the whole story. An agent that scores 1000 in 50
actions is far more efficient than one that needs 500 actions for the same result.
We define **efficiency = final_reward / total_actions** and compare winners vs losers
across all parsed episodes using a violin + strip chart.


In [ ]:
eff_records = []

for fp in valid:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        rew   = ep.get('rewards', [0, 0])
        ep_steps = ep.get('steps', [])
        if len(rew) < 2:
            continue

        for ag_idx in (0, 1):
            total_actions = sum(
                len((s[ag_idx].get('action') or {}))
                for s in ep_steps
                if len(s) > ag_idx
            )
            if total_actions == 0:
                continue
            eff_records.append({
                'agent':      f'Agent {ag_idx}',
                'reward':     rew[ag_idx],
                'actions':    total_actions,
                'efficiency': rew[ag_idx] / total_actions,
                'outcome':    'Winner' if rew[ag_idx] == max(rew) else 'Loser',
            })
    except Exception:
        continue

df_eff = pd.DataFrame(eff_records)
print(f'Efficiency records: {len(df_eff)}')

if not df_eff.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)

    palette = {'Winner': GREEN, 'Loser': CORAL}

    sns.violinplot(data=df_eff, x='outcome', y='efficiency',
                   palette=palette, inner=None, cut=0, ax=axes[0])
    sns.stripplot(data=df_eff, x='outcome', y='efficiency',
                  palette=palette, size=3, alpha=0.5, jitter=True, ax=axes[0])
    axes[0].set_title('Efficiency score -- winners vs losers')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Reward per action')
    axes[0].grid(True, axis='y')

    # Scatter: actions vs reward, colored by outcome
    for outcome, grp in df_eff.groupby('outcome'):
        axes[1].scatter(grp['actions'], grp['reward'],
                        c=palette[outcome], s=30, alpha=0.5,
                        label=outcome, edgecolors='none')
    axes[1].set_xlabel('Total actions issued')
    axes[1].set_ylabel('Final reward')
    axes[1].set_title('Actions vs reward -- efficiency frontier')
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle('Agent efficiency analysis', color=TEXT, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    print('\nEfficiency by outcome:')
    print(df_eff.groupby('outcome')['efficiency'].describe().round(4))


---
## 7. Action Markov chain

What action tends to follow what? Building a first-order transition matrix
over the action sequence for each agent reveals habitual patterns -- does the
agent alternate actions, repeat the same command, or have preferred sequences?
A normalized row-stochastic heatmap makes the dominant transitions clear.


In [ ]:
def build_transition_matrix(action_seq):
    """Return a row-normalised transition matrix and sorted action list."""
    actions = sorted(set(action_seq))
    idx = {a: i for i, a in enumerate(actions)}
    n = len(actions)
    mat = np.zeros((n, n), dtype=float)
    for prev, nxt in zip(action_seq[:-1], action_seq[1:]):
        mat[idx[prev], idx[nxt]] += 1
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return mat / row_sums, actions

if a0_actions and a1_actions:
    mat0, acts0 = build_transition_matrix(a0_actions)
    mat1, acts1 = build_transition_matrix(a1_actions)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)

    cmap = 'YlOrRd'
    for ax, mat, acts, title, border_c in [
        (axes[0], mat0, acts0, 'Agent 0 -- action transition matrix', CYAN),
        (axes[1], mat1, acts1, 'Agent 1 -- action transition matrix', CORAL),
    ]:
        im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(len(acts)))
        ax.set_yticks(range(len(acts)))
        ax.set_xticklabels(acts, rotation=40, ha='right', fontsize=8)
        ax.set_yticklabels(acts, fontsize=8)
        ax.set_xlabel('Next action')
        ax.set_ylabel('Current action')
        ax.set_title(title)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_c)
            spine.set_linewidth(1.8)
        for i in range(len(acts)):
            for j in range(len(acts)):
                v = mat[i, j]
                if v > 0.05:
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                            fontsize=7, color='black' if v > 0.4 else 'white')
        plt.colorbar(im, ax=ax, label='Transition probability')

    plt.tight_layout()
    plt.show()
else:
    print('No action data available for Markov chain.')


---
## 8. Early-game dominance signal

Does the agent ahead at 25% of the game nearly always win?
We extract the reward delta at the first quarter of each episode and compare
it to the final outcome -- yielding an early-dominance prediction accuracy metric.
A positive delta means Agent 0 was ahead early; negative means Agent 1 was.


In [ ]:
early_records = []

for fp in valid:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        rew   = ep.get('rewards', [0, 0])
        ep_st = ep.get('steps', [])
        if len(rew) < 2 or len(ep_st) < 8:
            continue

        q1 = max(1, int(len(ep_st) * 0.25))
        r0_q1 = ep_st[q1][0].get('reward') or 0 if len(ep_st[q1]) > 0 else 0
        r1_q1 = ep_st[q1][1].get('reward') or 0 if len(ep_st[q1]) > 1 else 0

        early_records.append({
            'early_delta': r0_q1 - r1_q1,
            'final_delta': rew[0] - rew[1],
            'a0_wins':     int(rew[0] > rew[1]),
        })
    except Exception:
        continue

df_early = pd.DataFrame(early_records)
print(f'Early-game records: {len(df_early)}')

if not df_early.empty:
    # Predictive accuracy: does sign(early_delta) == sign(final_delta)?
    df_early['correct'] = (
        np.sign(df_early['early_delta']) == np.sign(df_early['final_delta'])
    )
    acc = df_early['correct'].mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)

    # Scatter: early delta vs final delta
    colors = [GREEN if c else CORAL for c in df_early['correct']]
    axes[0].scatter(df_early['early_delta'], df_early['final_delta'],
                    c=colors, s=35, alpha=0.65, edgecolors='none')
    axes[0].axhline(0, color=BORDER, linewidth=1, linestyle='--')
    axes[0].axvline(0, color=BORDER, linewidth=1, linestyle='--')

    fit = np.polyfit(df_early['early_delta'], df_early['final_delta'], 1)
    xs = np.linspace(df_early['early_delta'].min(),
                     df_early['early_delta'].max(), 100)
    axes[0].plot(xs, np.polyval(fit, xs),
                 color=AMBER, linewidth=2, linestyle='-', label='Linear fit')

    axes[0].set_xlabel('Reward delta at 25% of game')
    axes[0].set_ylabel('Final reward delta')
    axes[0].set_title(f'Early vs final delta (accuracy {acc:.1f}%)')
    axes[0].legend()
    axes[0].grid(True)

    patch_ok  = mpatches.Patch(color=GREEN, label='Correct prediction')
    patch_bad = mpatches.Patch(color=CORAL, label='Wrong prediction')
    axes[0].legend(handles=[patch_ok, patch_bad])

    # Bar: correct vs incorrect
    counts = df_early['correct'].value_counts().reindex([True, False], fill_value=0)
    axes[1].bar(['Correct', 'Incorrect'],
                [counts[True], counts[False]],
                color=[GREEN, CORAL], width=0.5, alpha=0.85)
    axes[1].set_ylabel('Episode count')
    axes[1].set_title('Early-game dominance prediction accuracy')
    axes[1].grid(True, axis='y')
    for i, v in enumerate([counts[True], counts[False]]):
        axes[1].text(i, v + 0.5, str(v), ha='center',
                     color=TEXT, fontsize=12, fontweight='bold')

    plt.suptitle(f'Q1 delta predicts winner with {acc:.1f}% accuracy',
                 color=TEXT, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    corr = df_early['early_delta'].corr(df_early['final_delta'])
    print(f'Pearson correlation (early vs final delta): {corr:.3f}')
    print(f'Early-game prediction accuracy: {acc:.1f}%')


---
## 9. Crystal depletion dynamics

Crystals are the primary scoring resource. How quickly does the global supply
deplete across multiple episodes? We aggregate per-step crystal counts and
plot the mean depletion curve with a standard-deviation band, then fit an
exponential decay to estimate the half-life of the crystal supply.


In [ ]:
crys_curves = []

for fp in valid[:50]:  # cap at 50 for speed
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        ep_steps = ep.get('steps', [])
        curve = []
        for step in ep_steps:
            if step:
                obs = step[0].get('observation', {})
                curve.append(len(obs.get('globalCrystals', {})))
        if len(curve) > 10:
            crys_curves.append(curve)
    except Exception:
        continue

print(f'Crystal curves loaded: {len(crys_curves)}')

if crys_curves:
    max_len = max(len(c) for c in crys_curves)
    padded  = np.array([c + [np.nan] * (max_len - len(c)) for c in crys_curves])

    mean_curve = np.nanmean(padded, axis=0)
    std_curve  = np.nanstd(padded,  axis=0)

    # Normalise to [0, 1] for half-life computation
    c0 = mean_curve[0] if mean_curve[0] > 0 else 1.0
    norm_curve = mean_curve / c0

    # Fit y = exp(-lambda * t) to find half-life
    valid_mask = np.isfinite(norm_curve) & (norm_curve > 0)
    xs_fit = np.where(valid_mask)[0]
    if len(xs_fit) > 4:
        coeffs = np.polyfit(xs_fit, np.log(norm_curve[valid_mask] + 1e-9), 1)
        lam    = -coeffs[0]
        half_life = np.log(2) / lam if lam > 0 else float('inf')
    else:
        half_life = float('nan')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)

    # Mean curve with std band
    xs = np.arange(max_len)
    axes[0].fill_between(xs,
                         mean_curve - std_curve,
                         mean_curve + std_curve,
                         alpha=0.18, color=GREEN, label='Mean +/- 1 std')
    axes[0].plot(xs, mean_curve, color=GREEN, linewidth=2.4,
                 label='Mean crystal count')
    if np.isfinite(half_life):
        hl_step = int(half_life)
        if 0 < hl_step < max_len:
            axes[0].axvline(hl_step, color=AMBER, linewidth=1.5,
                            linestyle='--',
                            label=f'Half-life ~ step {hl_step}')
    axes[0].set_xlabel('Match step')
    axes[0].set_ylabel('Global crystal count')
    axes[0].set_title('Crystal depletion -- mean curve with std band')
    axes[0].legend()
    axes[0].grid(True, axis='y')

    # Individual curves (faint) + mean overlay
    for c in crys_curves[:20]:
        axes[1].plot(c, color=CYAN, alpha=0.12, linewidth=0.8)
    axes[1].plot(xs[:len(mean_curve)], mean_curve,
                 color=TEXT, linewidth=2.4, label='Mean')
    axes[1].set_xlabel('Match step')
    axes[1].set_ylabel('Crystal count')
    axes[1].set_title('Individual episode depletion curves (first 20)')
    axes[1].legend()
    axes[1].grid(True, axis='y')

    plt.suptitle('Crystal supply depletion dynamics', color=TEXT,
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    if np.isfinite(half_life):
        print(f'Estimated crystal half-life: {half_life:.1f} steps')
    print(f'Average crystal count at game start: {c0:.0f}')
    print(f'Average crystal count at game end:   {mean_curve[~np.isnan(mean_curve)][-1]:.0f}')


---
## 10. Match duration and win-rate analysis

Before strategic deep-dives, let's establish two population-level facts:
how long do matches typically last, and is there a systematic home-field
advantage for Agent 0 (the first agent in each replay file)?


In [ ]:
if not df_agg.empty:
    fig = plt.figure(figsize=(14, 6))
    fig.patch.set_facecolor(BG)
    gs = GridSpec(1, 2, figure=fig, wspace=0.38)

    # -- Match duration histogram with KDE overlay ------------------------
    ax0 = fig.add_subplot(gs[0])
    n_bins = min(30, max(10, len(df_agg) // 5))
    ax0.hist(df_agg['steps'], bins=n_bins, color=CYAN, alpha=0.55,
             edgecolor=BORDER, linewidth=0.5, density=True)
    sns.kdeplot(df_agg['steps'], color=CYAN, linewidth=2.4, ax=ax0)
    ax0.axvline(df_agg['steps'].mean(), color=AMBER, linewidth=1.8,
                linestyle='--', label=f'Mean {df_agg["steps"].mean():.0f} steps')
    ax0.axvline(df_agg['steps'].median(), color=GREEN, linewidth=1.8,
                linestyle=':', label=f'Median {df_agg["steps"].median():.0f} steps')
    ax0.set_xlabel('Match duration (steps)')
    ax0.set_ylabel('Density')
    ax0.set_title('Match duration distribution')
    ax0.legend()
    ax0.grid(True, axis='y')

    # -- Agent 0 win rate with Wilson confidence interval -----------------
    ax1 = fig.add_subplot(gs[1])
    n_total = len(df_agg)
    n_a0_wins = df_agg['a0_wins'].sum()
    p_hat = n_a0_wins / n_total if n_total > 0 else 0.5
    z = 1.96
    denom = 1 + z**2 / n_total
    centre = (p_hat + z**2 / (2 * n_total)) / denom
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / n_total + z**2 / (4 * n_total**2))) / denom
    ci_lo, ci_hi = max(0, centre - margin), min(1, centre + margin)

    agents = ['Agent 0', 'Agent 1']
    wins   = [n_a0_wins, n_total - n_a0_wins]
    bars = ax1.bar(agents, wins, color=[CYAN, CORAL], width=0.45, alpha=0.85)
    ax1.set_ylabel('Win count')
    ax1.set_title(
        f'Agent 0 win rate: {p_hat*100:.1f}% '
        f'(95% CI {ci_lo*100:.1f}--{ci_hi*100:.1f}%)'
    )
    ax1.grid(True, axis='y')
    for bar, v in zip(bars, wins):
        ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.3,
                 str(v), ha='center', color=TEXT, fontsize=12, fontweight='bold')
    ax1.axhline(n_total / 2, color=BORDER, linewidth=1.2,
                linestyle='--', label='50/50 baseline')
    ax1.legend()

    plt.suptitle(f'Population statistics across {n_total} episodes',
                 color=TEXT, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    print(f'Agent 0 wins: {n_a0_wins}/{n_total} ({p_hat*100:.1f}%)')
    print(f'95% Wilson CI: [{ci_lo*100:.1f}%, {ci_hi*100:.1f}%]')
else:
    print('No aggregate data to analyse.')


---
## 11. Robot energy distribution at game phases

Energy is the fuel that lets robots move and mine. We sample three game phases
(early = first 20%, mid = 40--60%, late = final 20%) and plot the distribution
of robot energy values. Narrowing distributions in the late phase signal that
agents with low-energy robots are eliminated, leaving only efficient survivors.


In [ ]:
def extract_energies(ep_steps, phase_frac_lo, phase_frac_hi):
    """Return list of robot energy values from steps in [phase_lo, phase_hi]."""
    n = len(ep_steps)
    lo = int(n * phase_frac_lo)
    hi = int(n * phase_frac_hi)
    energies = []
    for step in ep_steps[lo:hi]:
        if step:
            obs = step[0].get('observation', {})
            for r_data in obs.get('robots', {}).values():
                if len(r_data) >= 4:
                    e = r_data[3]
                    if isinstance(e, (int, float)) and e >= 0:
                        energies.append(float(e))
    return energies

early_e, mid_e, late_e = [], [], []

for fp in valid[:60]:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        ep_st = ep.get('steps', [])
        if len(ep_st) < 10:
            continue
        early_e.extend(extract_energies(ep_st, 0.0,  0.2))
        mid_e.extend(  extract_energies(ep_st, 0.4,  0.6))
        late_e.extend( extract_energies(ep_st, 0.8,  1.0))
    except Exception:
        continue

print(f'Energy samples -- early: {len(early_e)}, mid: {len(mid_e)}, late: {len(late_e)}')

if early_e and mid_e and late_e:
    phases = [
        ('Early (0--20%)', early_e, CYAN),
        ('Mid   (40--60%)', mid_e,  AMBER),
        ('Late  (80--100%)', late_e, CORAL),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
    fig.patch.set_facecolor(BG)

    for ax, (label, data, color) in zip(axes, phases):
        arr = np.array(data)
        q25, q50, q75 = np.percentile(arr, [25, 50, 75])
        ax.hist(arr, bins=30, color=color, alpha=0.55,
                edgecolor=BORDER, linewidth=0.4, density=True)
        try:
            sns.kdeplot(arr, color=color, linewidth=2.2, ax=ax)
        except Exception:
            pass
        ax.axvline(q50, color=TEXT, linewidth=1.6,
                   linestyle='--', label=f'Median {q50:.0f}')
        ax.set_title(label)
        ax.set_xlabel('Robot energy')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9)
        ax.grid(True, axis='y')
        ax.text(0.97, 0.95, f'IQR {q25:.0f}--{q75:.0f}',
                transform=ax.transAxes, ha='right', va='top',
                color=MUTED, fontsize=8)

    plt.suptitle('Robot energy distribution by game phase',
                 color=TEXT, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough energy data extracted -- check observation schema.')


---
## 12. Spatial activity heatmap

Where do robots actually spend their time? Aggregating robot (x, y) positions
across all parsed episodes produces a 2-D density map of the board. Hot zones
indicate contested or resource-rich regions; cold zones are strategically ignored.
We overlay the mean crystal density as contour lines to check whether robots
are tracking the resource distribution.


In [ ]:
robot_xy   = []
crystal_xy = []

for fp in valid[:40]:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
        for step in ep.get('steps', []):
            if not step:
                continue
            obs = step[0].get('observation', {})
            for r_data in obs.get('robots', {}).values():
                if len(r_data) >= 3:
                    robot_xy.append((float(r_data[1]), float(r_data[2])))
            for k in obs.get('globalCrystals', {}).keys():
                try:
                    x, y = map(int, k.split(','))
                    crystal_xy.append((float(x), float(y)))
                except ValueError:
                    pass
    except Exception:
        continue

print(f'Robot positions: {len(robot_xy):,}  |  Crystal positions: {len(crystal_xy):,}')

if len(robot_xy) > 50:
    rx = np.array([p[0] for p in robot_xy])
    ry = np.array([p[1] for p in robot_xy])

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.patch.set_facecolor(BG)

    # -- Robot density heatmap --------------------------------------------
    h = axes[0].hist2d(rx, ry,
                        bins=40,
                        cmap='plasma',
                        density=True)
    plt.colorbar(h[3], ax=axes[0], label='Visit density')
    axes[0].set_title('Robot spatial activity heatmap (all episodes)')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')
    axes[0].invert_yaxis()

    # -- Crystal density overlay (hexbin) ----------------------------------
    if len(crystal_xy) > 20:
        cx_arr = np.array([p[0] for p in crystal_xy])
        cy_arr = np.array([p[1] for p in crystal_xy])
        hb = axes[1].hexbin(cx_arr, cy_arr, gridsize=20,
                             cmap='YlGn', mincnt=1)
        plt.colorbar(hb, ax=axes[1], label='Crystal frequency')
        axes[1].set_title('Crystal position density (all episodes)')
        axes[1].set_xlabel('X coordinate')
        axes[1].set_ylabel('Y coordinate')
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'Insufficient crystal data',
                     ha='center', va='center', color=MUTED, transform=axes[1].transAxes)
        axes[1].set_title('Crystal position density')

    plt.suptitle('Spatial density analysis -- where the game is played',
                 color=TEXT, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough spatial data -- attach more daily episode datasets.')


---
## 13. Feature correlation matrix

A quick correlation scan across all numeric episode-level features helps identify
which variables move together. Strong correlations with `margin` or `winner` score
suggest promising predictive features for a downstream ML model.


In [ ]:
if not df_agg.empty and len(df_agg) > 5:
    numeric_cols = ['steps', 'a0', 'a1', 'winner', 'loser', 'margin', 'a0_wins']
    corr = df_agg[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor(BG)

    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    sns.heatmap(
        corr,
        mask=mask,
        cmap=cmap,
        vmin=-1, vmax=1, center=0,
        annot=True, fmt='.2f',
        linewidths=0.5, linecolor=BORDER,
        square=True, ax=ax,
        annot_kws={'size': 9, 'color': TEXT},
    )
    ax.set_facecolor(PANEL)
    ax.set_title('Episode-level feature correlation matrix', color=TEXT, pad=14)
    ax.tick_params(colors=MUTED)
    plt.tight_layout()
    plt.show()

    top_corr = (
        corr['margin'].drop('margin').abs()
             .sort_values(ascending=False)
    )
    print('Strongest correlations with score margin:')
    print(top_corr.round(3).to_string())
else:
    print('Need more episodes for a meaningful correlation matrix.')


---
## 14. Conclusion

This notebook covered seven layers of analysis across the Maze Crawler ecosystem:

**Macro trends** -- episode volume and storage grew steadily over 26 days;
the widening top-median gap signals an increasingly bifurcated competitive field.

**Episode dynamics** -- reward trajectories reveal clear exploitation phases.
Crystal and mining-node counts deplete monotonically, confirming a resource-scarcity
endgame where late-game positioning is decisive.

**Efficiency** -- winners achieve higher reward-per-action ratios, confirming that
action economy -- not raw activity -- is the primary success factor.

**Strategic patterns** -- action Markov chains show strong mining self-loops for
winners. Early-game dominance (Q1 delta) predicts the final outcome with high accuracy.

**Population stats** -- match durations cluster around a consistent length;
no systematic home-field advantage for Agent 0 was observed.

**Spatial dynamics** -- robots concentrate in resource-dense map zones;
crystal depletion curves suggest a half-life well within the match duration,
meaning late-game play is fundamentally about mining-node control.

**Next steps** -- combine spatial features, early-delta, and efficiency scores
into an offline RL or imitation-learning policy trained on top-agent replays.
